# 04 — HuggingFace Datasets Integration

This notebook covers:
- Loading a `datasets.Dataset` as an `InternalsDataset` with `from_hf_dataset()`
- Per-row spans from a dataset column
- Exporting extracted records to a `datasets.Dataset` with `to_hf_dataset()`
- Saving to disk and pushing to the Hub

**Requirements:** `pip install datasets transformers torch internals-extraction`

In [ ]:
import internals_extraction
from internals_extraction import (
    InternalsDataset, TextSpan, SpanSpec,
    to_hf_dataset, dump, load,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

MODEL     = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()
print("Ready")

## 1. Build a small HF dataset to work with

In a real workflow this would be `load_dataset("sst2", split="validation[:200]")` etc.

In [ ]:
from datasets import Dataset

raw = [
    {"sentence": "The Eiffel Tower is located in Paris.",
     "label": 1, "idx": 0,
     "span_annotations": {"entity": "The Eiffel Tower", "location": "Paris"}},
    {"sentence": "Albert Einstein was born in Ulm.",
     "label": 1, "idx": 1,
     "span_annotations": {"entity": "Albert Einstein", "location": "Ulm"}},
    {"sentence": "Python was created by Guido van Rossum.",
     "label": 1, "idx": 2,
     "span_annotations": {"entity": "Python", "person": "Guido van Rossum"}},
    {"sentence": "The sky is blue.",
     "label": 0, "idx": 3,
     "span_annotations": {}},
    {"sentence": "Dogs are loyal animals.",
     "label": 0, "idx": 4,
     "span_annotations": {"subject": "Dogs"}},
]

hf_ds = Dataset.from_list(raw)
print(hf_ds)
print(hf_ds[0])

## 2. `from_hf_dataset()` — uniform spans for all rows

In [ ]:
# Apply the same spans to every row
dataset = InternalsDataset.from_hf_dataset(
    hf_ds,
    text_col="sentence",
    property_cols=["label", "idx"],
)

print(f"Dataset length: {len(dataset)}")
print(f"Instance 0 text: {dataset[0].text!r}")
print(f"Instance 0 properties: {dataset[0].properties}")
print(f"Instance 0 spans: {dataset[0].spans}")

## 3. `from_hf_dataset()` — per-row span annotations from a column

If your dataset has a column of per-row annotations (substring strings, token-index lists, or `{start, end}` dicts), pass `spans_col=` to pick it up automatically.

In [ ]:
dataset_with_spans = InternalsDataset.from_hf_dataset(
    hf_ds,
    text_col="sentence",
    property_cols=["label", "idx"],
    spans_col="span_annotations",    # each cell: {name: "substring"} or {name: [s,e]}
)

print("Instances with their spans:")
for inst in dataset_with_spans:
    span_names = list(inst.spans or {})
    print(f"  [{inst.properties['idx']}] {inst.text[:40]:<42}  spans={span_names}")

## 4. Run the dataset

In [ ]:
records = dataset_with_spans.run(
    model,
    tokenizer,
    generate_kwargs={"max_new_tokens": 1},
    verbose=True,
)

In [ ]:
# Inspect one record
rec = records[0]
print(rec)
print()
print("properties:     ", rec.properties)
print("resolved_spans: ", rec.resolved_spans)
if rec.span_hidden_states_mean:
    for name, arr in rec.span_hidden_states_mean.items():
        print(f"span_{name}: {arr.shape}")

## 5. `to_hf_dataset()` — export to a HuggingFace Dataset

Converts each record into a row. Properties become top-level columns; array signals become nested lists; resolved span boundaries become `span_{name}_start` / `span_{name}_end` integer columns.

In [ ]:
out_ds = to_hf_dataset(records)
print(out_ds)
print()
print("Columns:", out_ds.column_names)

In [ ]:
# Array columns are stored as nested lists — convert back to numpy for analysis
row = out_ds[0]
print("Row keys:", list(row.keys()))
print()

hs_mean = np.array(row["input_hidden_states_mean"])   # (num_layers, hidden)
print(f"input_hidden_states_mean shape: {hs_mean.shape}")

# Check span boundary columns (only present if spans were resolved)
span_cols = [c for c in out_ds.column_names if c.startswith("span_")]
print(f"Span columns: {span_cols}")

if "span_entity" in out_ds.column_names:
    entity_arr = np.array(row["span_entity"])         # (num_layers, hidden)
    print(f"span_entity shape: {entity_arr.shape}")
    print(f"span_entity_start: {row.get('span_entity_start')}")
    print(f"span_entity_end:   {row.get('span_entity_end')}")

## 6. Filtering and slicing the output dataset

In [ ]:
# Filter to only factual instances (label == 1)
factual_ds = out_ds.filter(lambda row: row["label"] == 1)
print(f"Factual instances: {len(factual_ds)} / {len(out_ds)}")

# Select only metadata + span arrays (drop large full HS if present)
keep_cols = ["idx", "label", "run_id", "input_len", "num_layers",
             "input_hidden_states_mean", "output_hidden_states_mean"]
keep_cols = [c for c in keep_cols if c in out_ds.column_names]
slim_ds = out_ds.select_columns(keep_cols)
print(f"Slim dataset columns: {slim_ds.column_names}")

## 7. Save to disk

In [ ]:
import tempfile, os

# Save as HF Dataset (Arrow format)
hf_out_dir = tempfile.mkdtemp(prefix="internals_hf_")
out_ds.save_to_disk(hf_out_dir)
print(f"Saved HF dataset to: {hf_out_dir}")
print(f"Files: {os.listdir(hf_out_dir)}")

# Reload
from datasets import load_from_disk
reloaded = load_from_disk(hf_out_dir)
print(f"Reloaded: {reloaded}")

## 8. Also dump as .npz + metadata.jsonl

Use `dump()` for NumPy-native storage that does not require the `datasets` library.

In [ ]:
npz_out_dir = tempfile.mkdtemp(prefix="internals_npz_")
dump(records, npz_out_dir)

arrays_list, metadata_list = load(npz_out_dir)
print(f"Loaded {len(arrays_list)} records from .npz")

import json
with open(os.path.join(npz_out_dir, "metadata.jsonl")) as f:
    for line in f:
        meta = json.loads(line)
        print(f"  idx={meta['properties']['idx']}  "
              f"spans={list(meta['spans'].keys())}  "
              f"arrays={list(meta['arrays'].keys())}")

## 9. Push to Hub (optional)

```python
# Log in first:
#   huggingface-cli login
# or:
#   from huggingface_hub import login; login()

out_ds.push_to_hub(
    "my-org/gpt2-relation-internals",
    private=True,
)
```

The resulting dataset card will document all columns automatically.

## 10. End-to-end pipeline summary

```python
import internals_extraction
from internals_extraction import InternalsDataset, to_hf_dataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model     = AutoModelForCausalLM.from_pretrained("gpt2").eval()

# 2. Load data
ds = load_dataset("sst2", split="validation[:500]")

# 3. Build InternalsDataset
dataset = InternalsDataset.from_hf_dataset(
    ds,
    text_col="sentence",
    property_cols=["label", "idx"],
)

# 4. Extract internals
records = dataset.run(model, tokenizer, generate_kwargs={"max_new_tokens": 1})

# 5. Export
hf_out = to_hf_dataset(records)
hf_out.push_to_hub("my-org/gpt2-sst2-internals")
```